# MVA v3 · NB04B — BUBR1 predictor calibration before N1002K
**v1.0.2 · Public-data-only · preregistered comparator labels · N1002K held out**

## Question

Can any readily available missense predictor distinguish **experimentally low-abundance BUBR1 substitutions**
from **experimentally WT-like/stable BUBR1 substitutions**?

Only if a predictor succeeds on the frozen BUBR1-specific calibration set may its N1002K score be used as
supporting mechanistic evidence.

## Frozen comparator set

### Low-abundance / destabilized (`y=1`)
Clinical MVA substitutions:
- R727C
- R814H
- L844F
- I909T
- L1012P

Engineered pseudokinase-domain stability controls:
- K795R
- D911A

### WT-like/stable (`y=0`)
Clinical/observed substitutions:
- Y155C
- R550Q
- Q921H

Engineered stability control:
- D882A

The engineered controls are marked explicitly and are never counted as clinical pathogenic variants.

## Frozen predictor gate

A predictor may be applied to N1002K only if, **without N1002K in calibration**:

1. ROC AUC >= **0.80**
2. bootstrap 95% AUC lower bound > **0.50**
3. leave-one-out balanced accuracy >= **0.70**

Metrics:
- AlphaMissense
- SIFT
- PolyPhen
- BLOSUM62
- ESM2-150M masked-marginal ΔlogP

No ensemble is trained: n=11 is too small for a defensible multivariable model.

## v1.0.1 technical corrections

- ESM2-150M accepts at most 1026 positions including special tokens. BUBR1 is 1050 aa, so masked-marginal
  scores are computed in a deterministic **1000-aa local window** rather than by passing the full sequence.
- Current Ensembl REST AlphaMissense output is parsed from `am_pathogenicity` / `am_class` as well as
  compatibility aliases.
- Missing predictor values remain missing; they are never imputed.
## v1.0.2 completeness correction

v1.0.1 correctly returned `NO_PREDICTOR_CALIBRATED`, but Ensembl protein-HGVS retrieval was incomplete for
`Q921H`, `D882A`, and the target `N1002K`. This version changes **only the technical retrieval route**.

The scientific comparator set and frozen thresholds are unchanged.

v1.0.2:
- retrieves the MANE Ensembl CDS (`ENST00000287598`);
- reconstructs the exact single-nucleotide `c.` HGVS that produces each amino-acid substitution;
- verifies reference codons against the canonical BUBR1 protein;
- queries VEP using coding HGVS rather than potentially ambiguous protein HGVS;
- retries transient 429/5xx responses;
- requires all 11 frozen comparator values before a VEP-based predictor is eligible to pass the frozen gate;
- reports target scores only after calibration.

No missing value is imputed.

In [ ]:
import os
# Public-data-only environment
import importlib.util, subprocess, sys
pkgs = {
    "requests":"requests==2.34.2",
    "pandas":"pandas==3.0.6",
    "numpy":"numpy==2.5.3",
    "sklearn":"scikit-learn==1.9.1",
    "torch":"torch==2.14.0",
    "transformers":"transformers==5.17.0",
    "Bio":"biopython==1.88",
}
for mod,pkg in pkgs.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

import json, math, re, time, requests
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from Bio.Align import substitution_matrices

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
DOC_OUT=ROOT/"04_DOCUMENTATION"
PUBLIC_OUT.mkdir(parents=True,exist_ok=True)
DOC_OUT.mkdir(parents=True,exist_ok=True)

UNIPROT="O60566"
ENST="ENST00000287598"
ENSP="ENSP00000287598"

## 1. Frozen experimental labels

In [ ]:
comparators = pd.DataFrame([
    # Clinical / disease-associated substitutions with direct experimental abundance/stability evidence.
    ("R727C",727,"R","C",1,"clinical","Suijkerbuijk2010","low abundance; ~2x increased turnover"),
    ("R814H",814,"R","H",1,"clinical","Suijkerbuijk2010","5-10x lower ectopic abundance class"),
    ("L844F",844,"L","F",1,"clinical","Suijkerbuijk2010","poorly expressed; Q921H co-allelic control WT-like"),
    ("I909T",909,"I","T",1,"clinical","Suijkerbuijk2010","low abundance; ~2x increased turnover; HSP90-dependent"),
    ("L1012P",1012,"L","P",1,"clinical","Suijkerbuijk2010;Sieben2020","low abundance; HSP90/proteasome; rescue at WT-like expression"),

    # Engineered pseudokinase-domain controls from the later structural study.
    ("K795R",795,"K","R",1,"engineered_control","Cordeiro2020","destabilized / lower protein levels"),
    ("D911A",911,"D","A",1,"engineered_control","Cordeiro2020","destabilized / lower protein levels"),

    # Stable / WT-like substitutions.
    ("Y155C",155,"Y","C",0,"clinical","Suijkerbuijk2010","did not reduce ectopic protein abundance; functional SAC defect"),
    ("R550Q",550,"R","Q",0,"clinical","Suijkerbuijk2010","WT-like abundance; rescued tested functions"),
    ("Q921H",921,"Q","H",0,"clinical","Suijkerbuijk2010","WT-like abundance and function"),
    ("D882A",882,"D","A",0,"engineered_control","Cordeiro2020","did not disrupt stability/expression"),
], columns=["variant","position","ref","alt","low_abundance","source_type","source","label_basis"])

assert len(comparators)==11
assert comparators.low_abundance.sum()==7
assert (comparators.low_abundance==0).sum()==4

target = pd.DataFrame([("N1002K",1002,"N","K")],columns=["variant","position","ref","alt"])

display(comparators)
print("N calibration =",len(comparators),"; low abundance =",comparators.low_abundance.sum(),
      "; stable =",sum(comparators.low_abundance==0))

## 2. Public sequence integrity check

In [ ]:
fa=requests.get(f"https://rest.uniprot.org/uniprotkb/{UNIPROT}.fasta",timeout=60)
fa.raise_for_status()
seq="".join(x.strip() for x in fa.text.splitlines() if not x.startswith(">"))
assert len(seq)==1050, f"Unexpected BUBR1 length: {len(seq)}"

for _,r in pd.concat([comparators,target],ignore_index=True).iterrows():
    observed=seq[int(r.position)-1]
    assert observed==r.ref, f"{r.variant}: reference mismatch, sequence has {observed}"

print("Canonical O60566 sequence integrity: PASS; length",len(seq))

## 3. Ensembl VEP public predictor retrieval

In [ ]:
# Only the 11 public comparator substitutions + public target N1002K are queried.
# No private VCF/genotype/sample identifier is transmitted.

from Bio.Seq import Seq

# Retrieve the MANE Select Ensembl CDS. Ensembl sequence/id supports type=cds.
CDS_URL=f"https://rest.ensembl.org/sequence/id/{ENST}"
cds_resp=requests.get(
    CDS_URL,
    headers={"Content-Type":"text/plain","Accept":"text/plain"},
    params={"type":"cds"},
    timeout=90
)
cds_resp.raise_for_status()
cds=re.sub(r"\s+","",cds_resp.text).upper()

assert len(cds) >= 3150, f"Unexpected CDS length: {len(cds)}"
translated=str(Seq(cds[:3150]).translate())
assert translated == seq, "MANE Ensembl CDS does not reproduce canonical 1050-aa BUBR1 sequence."
print("MANE CDS -> canonical protein integrity: PASS")

DNA="ACGT"

def derive_single_snv_routes(row):
    """All one-nucleotide CDS substitutions producing the requested AA change."""
    pos=int(row.position)
    ref_aa=str(row.ref); alt_aa=str(row.alt)
    codon_start=(pos-1)*3
    codon=cds[codon_start:codon_start+3]
    assert len(codon)==3
    assert str(Seq(codon).translate())==ref_aa, (row.variant,codon,ref_aa)

    routes=[]
    for j in range(3):
        ref_nt=codon[j]
        for alt_nt in DNA:
            if alt_nt==ref_nt:
                continue
            mut=list(codon)
            mut[j]=alt_nt
            mut="".join(mut)
            if str(Seq(mut).translate())==alt_aa:
                cpos=codon_start+j+1
                routes.append({
                    "hgvs":f"{ENST}:c.{cpos}{ref_nt}>{alt_nt}",
                    "cpos":cpos,
                    "ref_nt":ref_nt,
                    "alt_nt":alt_nt,
                    "ref_codon":codon,
                    "alt_codon":mut,
                })
    assert routes, f"{row.variant}: no single-nucleotide route found."
    return routes

def recursive_find(obj, keys):
    found={}
    if isinstance(obj,dict):
        for k,v in obj.items():
            kl=k.lower()
            for wanted in keys:
                if kl==wanted.lower() and wanted not in found:
                    found[wanted]=v
            sub=recursive_find(v,keys)
            for k,v in sub.items():
                found.setdefault(k,v)
    elif isinstance(obj,list):
        for x in obj:
            sub=recursive_find(x,keys)
            for k,v in sub.items():
                found.setdefault(k,v)
    return found

session=requests.Session()

def query_vep_hgvs(hgvs, max_attempts=6):
    url="https://rest.ensembl.org/vep/human/hgvs/"+requests.utils.quote(hgvs,safe=":._>")
    params={"AlphaMissense":1,"Blosum62":1}
    last_error=None
    for attempt in range(max_attempts):
        try:
            rr=session.get(
                url,
                headers={"Content-Type":"application/json","Accept":"application/json"},
                params=params,
                timeout=90
            )
            if rr.ok:
                data=rr.json()
                vals=recursive_find(data,[
                    "am_pathogenicity","am_class",
                    "alphamissense","alphamissense_pred","alphamissense_score",
                    "sift_score","sift_prediction",
                    "polyphen_score","polyphen_prediction",
                    "blosum62"
                ])
                vals["query_hgvs"]=hgvs
                vals["http_status"]=rr.status_code
                return vals
            last_error=f"{rr.status_code}: {rr.text[:180]}"
            if rr.status_code not in (429,500,502,503,504):
                break
        except Exception as e:
            last_error=f"{type(e).__name__}: {e}"
        time.sleep(min(2**attempt,16))
    return {"query_hgvs":hgvs,"vep_error":last_error}

def num(x):
    if x is None: return np.nan
    if isinstance(x,(float,int,np.number)): return float(x)
    m=re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",str(x))
    return float(m.group()) if m else np.nan

def route_triplet(vals):
    return {
        "AlphaMissense":num(vals.get("am_pathogenicity",
                         vals.get("alphamissense_score",
                         vals.get("alphamissense")))),
        "SIFT":num(vals.get("sift_score")),
        "PolyPhen":num(vals.get("polyphen_score")),
    }

def resolve_variant(row):
    routes=derive_single_snv_routes(row)
    queried=[]
    for route in routes:
        vals=query_vep_hgvs(route["hgvs"])
        rec={**route,**vals,**route_triplet(vals)}
        queried.append(rec)
        time.sleep(0.15)

    q=pd.DataFrame(queried)
    complete=q[["AlphaMissense","SIFT","PolyPhen"]].notna().all(axis=1)

    out={
        "variant":row.variant,
        "n_coding_snv_routes":len(routes),
        "coding_routes":";".join(r["hgvs"] for r in routes),
        "n_complete_routes":int(complete.sum()),
        "route_errors":" | ".join(
            str(x) for x in q.get("vep_error",pd.Series(dtype=object)).dropna().tolist()
        )
    }

    # For an amino-acid predictor, different nucleotide routes to the same substitution should agree.
    # If they do not, the comparator is technically route-sensitive and is left missing.
    for metric in ["AlphaMissense","SIFT","PolyPhen"]:
        vals=q.loc[np.isfinite(q[metric]),metric].to_numpy(float)
        if len(vals)==0:
            out[metric]=np.nan
            out[metric+"_route_consistent"]=False
        elif np.ptp(vals) <= 1e-12:
            out[metric]=float(vals[0])
            out[metric+"_route_consistent"]=True
        else:
            out[metric]=np.nan
            out[metric+"_route_consistent"]=False
    return out,queried

allv=pd.concat([
    comparators[["variant","position","ref","alt"]],
    target[["variant","position","ref","alt"]]
],ignore_index=True)

resolved=[]
route_audit=[]
for _,r in allv.iterrows():
    out,details=resolve_variant(r)
    resolved.append(out)
    for d in details:
        route_audit.append({"variant":r.variant,**d})

vep_resolved=pd.DataFrame(resolved)
route_audit=pd.DataFrame(route_audit)

display(vep_resolved)
vep_resolved.to_csv(PUBLIC_OUT/"NB04B_v102_resolved_predictors.csv",index=False)
route_audit.to_csv(PUBLIC_OUT/"NB04B_v102_coding_route_audit.csv",index=False)

# Hard QA: the target must now have all three public VEP-based scores.
tqa=vep_resolved.set_index("variant").loc["N1002K",["AlphaMissense","SIFT","PolyPhen"]]
assert tqa.notna().all(), f"N1002K predictor retrieval still incomplete: {tqa.to_dict()}"
print("N1002K VEP predictor completeness: PASS")

## 4. BLOSUM62

In [ ]:
blosum=substitution_matrices.load("BLOSUM62")
comparators["BLOSUM62"]=comparators.apply(lambda r:float(blosum[r.ref,r.alt]),axis=1)
target["BLOSUM62"]=target.apply(lambda r:float(blosum[r.ref,r.alt]),axis=1)
display(comparators[["variant","low_abundance","BLOSUM62"]])
print("N1002K BLOSUM62:",target.loc[0,"BLOSUM62"])

## 5. ESM2-150M masked-marginal ΔlogP

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME="facebook/esm2_t30_150M_UR50D"
device="cuda" if torch.cuda.is_available() else "cpu"
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

# ESM2-150M max_position_embeddings=1026. With CLS/EOS, use a 1000-aa sequence window.
ESM_WINDOW=1000

def local_window(full_seq, pos1, window=ESM_WINDOW):
    """Return deterministic <=window-aa subsequence and 1-based local position."""
    L=len(full_seq)
    assert 1 <= pos1 <= L
    if L <= window:
        return full_seq, pos1, 1, L

    half=window//2
    start=max(1, pos1-half)
    end=start+window-1
    if end>L:
        end=L
        start=L-window+1
    local_pos=pos1-start+1
    assert 1 <= local_pos <= len(full_seq[start-1:end])
    return full_seq[start-1:end], local_pos, start, end

@torch.no_grad()
def esm_delta_logp(pos,ref,alt):
    subseq, local_pos, start, end = local_window(seq, int(pos))
    chars=list(subseq)
    assert chars[local_pos-1]==ref, (pos,ref,chars[local_pos-1],start,end)
    chars[local_pos-1]=tok.mask_token

    inp=tok("".join(chars),return_tensors="pt",add_special_tokens=True).to(device)
    seq_len=int(inp["input_ids"].shape[1])
    assert seq_len <= 1026, f"ESM input too long: {seq_len}"

    logits=model(**inp).logits[0]
    ids=inp["input_ids"][0]
    mask_idx=(ids==tok.mask_token_id).nonzero(as_tuple=True)[0]
    assert len(mask_idx)==1

    lp=torch.log_softmax(logits[mask_idx.item()],dim=-1)
    ref_id=tok.convert_tokens_to_ids(ref)
    alt_id=tok.convert_tokens_to_ids(alt)
    assert ref_id != tok.unk_token_id and alt_id != tok.unk_token_id
    return float((lp[alt_id]-lp[ref_id]).cpu())

comparators["ESM2_dlogp"]=comparators.apply(
    lambda r:esm_delta_logp(int(r.position),r.ref,r.alt),axis=1)
target["ESM2_dlogp"]=target.apply(
    lambda r:esm_delta_logp(int(r.position),r.ref,r.alt),axis=1)

display(comparators[["variant","low_abundance","ESM2_dlogp"]])
print("N1002K ESM2 ΔlogP:",target.loc[0,"ESM2_dlogp"])

## 6. Normalize retrieved VEP fields

In [ ]:
rmap=vep_resolved.set_index("variant")

for table in [comparators,target]:
    table["AlphaMissense"]=table.variant.map(lambda v:rmap.loc[v,"AlphaMissense"])
    table["SIFT"]=table.variant.map(lambda v:rmap.loc[v,"SIFT"])
    table["PolyPhen"]=table.variant.map(lambda v:rmap.loc[v,"PolyPhen"])

# Direction: higher = more likely experimentally low abundance.
comparators["SIFT_harm"]=1-comparators["SIFT"]
target["SIFT_harm"]=1-target["SIFT"]
comparators["minus_BLOSUM62"]=-comparators["BLOSUM62"]
target["minus_BLOSUM62"]=-target["BLOSUM62"]
comparators["minus_ESM2_dlogp"]=-comparators["ESM2_dlogp"]
target["minus_ESM2_dlogp"]=-target["ESM2_dlogp"]

display(comparators)
display(target)

## 7. Frozen calibration test

In [ ]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

PREDICTORS=[
    "AlphaMissense",
    "SIFT_harm",
    "PolyPhen",
    "minus_BLOSUM62",
    "minus_ESM2_dlogp",
]

def auc_oriented(y,x):
    ok=np.isfinite(x)
    if ok.sum()<6 or len(np.unique(y[ok]))<2:
        return np.nan
    return roc_auc_score(y[ok],x[ok])

def bootstrap_auc(y,x,B=5000,seed=20260922):
    y=np.asarray(y); x=np.asarray(x,float)
    ok=np.isfinite(x); y=y[ok]; x=x[ok]
    rng=np.random.default_rng(seed)
    vals=[]
    for _ in range(B):
        idx=rng.integers(0,len(y),len(y))
        yy=y[idx]; xx=x[idx]
        if len(np.unique(yy))<2: continue
        vals.append(roc_auc_score(yy,xx))
    return np.quantile(vals,[0.025,0.5,0.975]) if vals else [np.nan]*3

def loo_balacc(y,x):
    y=np.asarray(y); x=np.asarray(x,float)
    ok=np.isfinite(x); y=y[ok]; x=x[ok]
    preds=[]; truth=[]
    for i in range(len(y)):
        train=np.arange(len(y))!=i
        yy=y[train]; xx=x[train]
        u=np.sort(np.unique(xx))
        th=np.r_[-np.inf,(u[:-1]+u[1:])/2,np.inf]
        scores=[]
        for t in th:
            pp=(xx>=t).astype(int)
            scores.append(balanced_accuracy_score(yy,pp))
        best=th[int(np.argmax(scores))]
        preds.append(int(x[i]>=best)); truth.append(int(y[i]))
    return balanced_accuracy_score(truth,preds)

y=comparators.low_abundance.to_numpy(int)
stats=[]
for p in PREDICTORS:
    x=comparators[p].to_numpy(float)
    ok=np.isfinite(x)
    n_used=int(ok.sum())
    n_low=int(np.sum(y[ok]==1))
    n_wt=int(np.sum(y[ok]==0))
    complete_frozen_set=(n_used==len(comparators))

    auc=auc_oriented(y,x)
    ci=bootstrap_auc(y,x)
    loo=loo_balacc(y,x) if n_used>=2 else np.nan

    # Scientific frozen thresholds are unchanged.
    # Completeness is a technical eligibility QA, not a new biological threshold.
    passed=bool(
        complete_frozen_set and
        np.isfinite(auc) and auc>=0.80 and
        ci[0]>0.50 and
        loo>=0.70
    )

    stats.append({
        "predictor":p,
        "n_used":n_used,
        "n_low_abundance_used":n_low,
        "n_WT_like_used":n_wt,
        "complete_frozen_set":complete_frozen_set,
        "auc":auc,
        "auc_ci_low":ci[0],
        "auc_ci_med":ci[1],
        "auc_ci_high":ci[2],
        "loo_balanced_accuracy":loo,
        "passes_frozen_gate":passed,
        "N1002K_score":float(target.loc[0,p]) if np.isfinite(target.loc[0,p]) else np.nan
    })

stats=pd.DataFrame(stats).sort_values(
    ["passes_frozen_gate","auc"],ascending=[False,False]
)
display(stats)
stats.to_csv(PUBLIC_OUT/"NB04B_predictor_calibration_v1_0_2.csv",index=False)

assert stats.complete_frozen_set.all(), (
    "v1.0.2 completeness QA failed: at least one predictor does not cover all 11 frozen comparators."
)
print("All predictor classes cover all 11 frozen comparator entries: PASS")

## 8. Interpret N1002K only after gate

In [ ]:
passed=stats[stats.passes_frozen_gate].copy()

if passed.empty:
    final="NO_PREDICTOR_CALIBRATED"
    target_interpretation=(
        "No tested predictor met the preregistered BUBR1 abundance-class calibration gate on the complete "
        "11-variant frozen comparator set. N1002K remains AMBIGUOUS for abundance limitation."
    )
else:
    final="ONE_OR_MORE_PREDICTORS_CALIBRATED"
    target_interpretation=(
        "Only predictors passing the frozen complete-set BUBR1 gate may contribute supporting abundance-mechanism evidence. "
        "These are not automatically ACMG PP3/BP4 evidence."
    )

handoff={
    "schema":"MVA-NB04B-BUBR1-PREDICTOR-CALIBRATION-1.0.2",
    "result":final,
    "supersedes_technical_result":"v1.0.1 because VEP-based predictors were incomplete for Q921H/D882A/N1002K",
    "scientific_thresholds_changed":False,
    "n_calibration_variants":len(comparators),
    "n_low_abundance":int(comparators.low_abundance.sum()),
    "n_WT_like":int((comparators.low_abundance==0).sum()),
    "complete_set_required":True,
    "frozen_gate":{
        "auc_min":0.80,
        "bootstrap_auc_ci_low_gt":0.50,
        "loo_balanced_accuracy_min":0.70
    },
    "passing_predictors":passed.predictor.tolist(),
    "target":"BUB1B N1002K",
    "target_scores":{
        r.predictor:(None if not np.isfinite(r.N1002K_score) else float(r.N1002K_score))
        for _,r in stats.iterrows()
    },
    "target_interpretation":target_interpretation,
    "important_limitations":[
        "Small n=11.",
        "Two destabilized and one stable entries are engineered assay controls, not clinical variants.",
        "Predictors measure generic substitution/pathogenicity or sequence plausibility rather than protein abundance specifically.",
        "ESM2 is evaluated in deterministic 1000-aa windows because full-length BUBR1 exceeds the model context limit.",
        "No multivariable model is trained to avoid severe overfitting."
    ],
    "privacy":{
        "private_VCF_used":False,
        "private_genotypes_used":False,
        "only_public_BUBR1_substitutions_queried":True
    }
}
(DOC_OUT/"NB04B_PREDICTOR_CALIBRATION_HANDOFF_v1_0_2.json").write_text(
    json.dumps(handoff,indent=2)
)
print(json.dumps(handoff,indent=2))